In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import tifffile as tiff
import matplotlib.pyplot as plt
from IPython.display import display
from skimage.measure import regionprops, label as cc_label
from skimage.morphology import binary_erosion, remove_small_holes

# =============================================================
# Manual Sarcomere NanoSIMS measurement notebook cell
#
# Behavior:
# - Loads the cell label TIFF
# - Reduces it to 256 x 256 by block-wise modal value
# - Uses the reduced cell mask as the cell-assignment image
# - Assigns each sarcomere ROI to the cell label at BX/BY in 256 x 256 space
# - Prints and saves ROI-to-cell assignments
# - Shows overlays with ROI -> Cell labels
# =============================================================

# -------------------------------------------------------------
# Paths
# -------------------------------------------------------------
INPUT_ROOT = Path("input")
OUTPUT_ROOT = Path("output_manual_sarcomere_measurements")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BASE_MORPHOLOGY_CANDIDATES = [
    Path("combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated.csv"),
    Path("/mnt/data/combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated.csv"),
]

BASE_MORPHOLOGY_PATH = None
for p in BASE_MORPHOLOGY_CANDIDATES:
    if p.exists():
        BASE_MORPHOLOGY_PATH = p
        break

if BASE_MORPHOLOGY_PATH is None:
    raise FileNotFoundError(
        "Could not find combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated.csv"
    )

print("INPUT_ROOT :", INPUT_ROOT.resolve())
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())
print("BASE_MORPHOLOGY_PATH:", BASE_MORPHOLOGY_PATH.resolve())

# -------------------------------------------------------------
# Settings
# -------------------------------------------------------------
ROI_HALF_WIDTH = 1  # 1 -> 3x3, 2 -> 5x5, 4 -> 9x9
COORDS_ARE_ONE_BASED = True
DISPLAY_CHANNEL_INDEX = 2  # third NanoSIMS channel (zero-based index 2)
PIXEL_SIZE_NM = 1.751

DEFAULT_LABEL = "Sarcomere"
DEFAULT_SOURCE_EM_FILE = ""

# Optional orientation fixes for the cell label image after loading.
CELL_FLIPUD = False
CELL_FLIPLR = False
CELL_ROTATE_K = 0

EXPECTED_CHANNELS_7 = ["16O", "12C21H", "12C14N", "12C15N", "29Si", "31P", "32S"]
EXPECTED_CHANNELS_6 = ["16O", "12C21H", "12C14N", "12C15N", "31P", "32S"]
CELL_CHANNEL_NAME = "cell_label"

MORPH_COLS = pd.read_csv(BASE_MORPHOLOGY_PATH, nrows=0).columns.tolist()

# -------------------------------------------------------------
# Helpers
# -------------------------------------------------------------
def infer_channel_names(n_channels):
    if n_channels == 7:
        return EXPECTED_CHANNELS_7.copy()
    if n_channels == 6:
        return EXPECTED_CHANNELS_6.copy()
    raise ValueError(f"Unsupported channel count: {n_channels}. Expected 6 or 7.")


def finite_values(a):
    a = np.asarray(a, dtype=float).ravel()
    return a[np.isfinite(a)]


def summarize(values):
    v = finite_values(values)
    if v.size == 0:
        return {
            "n": 0, "mean": np.nan, "median": np.nan, "std": np.nan,
            "min": np.nan, "max": np.nan, "q1": np.nan, "q3": np.nan, "iqr": np.nan
        }
    q1 = float(np.percentile(v, 25))
    q3 = float(np.percentile(v, 75))
    return {
        "n": int(v.size),
        "mean": float(np.mean(v)),
        "median": float(np.median(v)),
        "std": float(np.std(v)),
        "min": float(np.min(v)),
        "max": float(np.max(v)),
        "q1": q1,
        "q3": q3,
        "iqr": float(q3 - q1),
    }


def add_stats(row, prefix, values):
    s = summarize(values)
    for k, v in s.items():
        row[f"{prefix}_{k}"] = v


def whole_field_mean_norm(stack, channel_names):
    n_channels = stack.shape[0]
    if len(channel_names) != n_channels:
        raise ValueError(f"Channel-name count ({len(channel_names)}) does not match stack channels ({n_channels}).")
    means = np.empty(n_channels, dtype=float)
    for ch, name in enumerate(channel_names):
        vals = stack[ch]
        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size == 0:
            means[ch] = 1.0
        else:
            mu = float(np.mean(finite_vals))
            means[ch] = mu if np.isfinite(mu) and mu != 0 else 1.0
    return stack / means[:, None, None], means


def whole_field_mean_norm_masked(stack, channel_names, mask):
    n_channels = stack.shape[0]
    if len(channel_names) != n_channels:
        raise ValueError(f"Channel-name count ({len(channel_names)}) does not match stack channels ({n_channels}).")
    mask = np.asarray(mask, dtype=bool)
    if mask.shape != stack.shape[1:]:
        raise ValueError(f"Mask shape {mask.shape} does not match stack spatial shape {stack.shape[1:]}.")
    means = np.empty(n_channels, dtype=float)
    for ch, name in enumerate(channel_names):
        vals = stack[ch][mask]
        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size == 0:
            means[ch] = 1.0
        else:
            mu = float(np.mean(finite_vals))
            means[ch] = mu if np.isfinite(mu) and mu != 0 else 1.0
    return stack / means[:, None, None], means


def find_first_tiff(folder: Path):
    files = sorted(list(folder.glob("*.tif")) + list(folder.glob("*.tiff")))
    if not files:
        raise FileNotFoundError(f"No TIFF files found in {folder}")
    if len(files) > 1:
        print(f"Warning: multiple TIFFs found in {folder}, using {files[0].name}")
    return files[0]


def load_stack(stack_path: Path):
    stack = tiff.imread(str(stack_path))
    if stack.ndim != 3:
        raise ValueError(f"Expected a 3D TIFF stack, got shape {stack.shape} for {stack_path.name}")
    if stack.shape[1:] == (256, 256):
        pass
    elif stack.shape[:2] == (256, 256):
        stack = np.moveaxis(stack, -1, 0)
    else:
        raise ValueError(f"Expected 256x256 spatial shape, got {stack.shape} for {stack_path.name}")
    if stack.shape[0] not in (6, 7):
        raise ValueError(f"Expected 6 or 7 channels, got {stack.shape[0]} in {stack_path.name}")
    return stack.astype(np.float64)


def parse_section_folder_name(section_name: str):
    s = str(section_name)
    m_sample = re.search(r"(?i)S0*(\d+)", s)
    m_area = re.search(r"(?i)Ar\s*0*(\d+)", s)
    if m_sample is None or m_area is None:
        raise ValueError(f"Could not parse sample/area from section folder name '{section_name}'.")
    sample = f"S{int(m_sample.group(1))}"
    area = f"Ar{int(m_area.group(1))}"
    return sample, area


def find_csv_files(csv_dir: Path):
    if not csv_dir.exists():
        return []
    return sorted(csv_dir.glob("*.csv"))


def clean_cell_labels(cell_img: np.ndarray):
    """
    Preserve positive label IDs exactly as they appear.
    Only remove tiny holes and keep the largest connected component per label.
    """
    cell_img = np.rint(cell_img).astype(np.int32)
    cleaned = np.zeros_like(cell_img, dtype=np.int32)

    for lab in np.unique(cell_img):
        lab = int(lab)
        if lab <= 0:
            continue

        mask = cell_img == lab
        if not np.any(mask):
            continue

        mask = remove_small_holes(mask, area_threshold=mask.size)
        cc = cc_label(mask, connectivity=2)
        if cc.max() == 0:
            continue

        counts = np.bincount(cc.ravel())
        counts[0] = 0
        keep = int(np.argmax(counts))
        cleaned[cc == keep] = lab

    return cleaned


def block_mode(values):
    """
    Return the modal value in a 1D array.
    Prefer positive labels over 0 when any positive label exists.
    """
    values = np.asarray(values).ravel().astype(np.int32)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return 0

    positive = values[values > 0]
    if positive.size > 0:
        values = positive

    vals, cnts = np.unique(values, return_counts=True)
    return int(vals[np.argmax(cnts)])


def downsample_label_image_to_target(label_img: np.ndarray, target_shape=(256, 256)):
    """
    Reduce a labeled cell image to target_shape by taking the modal label
    within each block.
    """
    if label_img.ndim != 2:
        raise ValueError(f"Expected 2D label image, got shape {label_img.shape}")

    src_h, src_w = label_img.shape
    tgt_h, tgt_w = target_shape

    y_edges = np.linspace(0, src_h, tgt_h + 1)
    x_edges = np.linspace(0, src_w, tgt_w + 1)

    out = np.zeros((tgt_h, tgt_w), dtype=np.int32)

    for i in range(tgt_h):
        y0 = int(np.floor(y_edges[i]))
        y1 = int(np.ceil(y_edges[i + 1]))
        y1 = min(max(y1, y0 + 1), src_h)

        for j in range(tgt_w):
            x0 = int(np.floor(x_edges[j]))
            x1 = int(np.ceil(x_edges[j + 1]))
            x1 = min(max(x1, x0 + 1), src_w)

            block = label_img[y0:y1, x0:x1]
            out[i, j] = block_mode(block)

    return out


def load_cell_labels(cell_path: Path, target_shape=(256, 256)):
    cell_img = tiff.imread(str(cell_path))
    if cell_img.ndim != 2:
        raise ValueError(f"Expected a 2D cell label TIFF, got shape {cell_img.shape} for {cell_path.name}")

    cell_img = np.rint(cell_img).astype(np.int32)

    if CELL_ROTATE_K:
        cell_img = np.rot90(cell_img, k=CELL_ROTATE_K)
    if CELL_FLIPUD:
        cell_img = np.flipud(cell_img)
    if CELL_FLIPLR:
        cell_img = np.fliplr(cell_img)

    cell_small = downsample_label_image_to_target(cell_img, target_shape=target_shape)
    return cell_img, cell_small


def append_cell_channel_to_stack(stack, cell_small):
    """
    Append the reduced cell label image as a new channel for visualization.
    """
    cell_channel = cell_small.astype(np.float64)[None, :, :]
    return np.concatenate([stack, cell_channel], axis=0)


def sample_center(bx, by):
    cx = int(round(float(bx)))
    cy = int(round(float(by)))
    if COORDS_ARE_ONE_BASED:
        cx -= 1
        cy -= 1
    return cx, cy


def measure_roi_mask(spatial_shape, bx, by, half_width=1):
    cx, cy = sample_center(bx, by)
    y0 = cy - half_width
    y1 = cy + half_width + 1
    x0 = cx - half_width
    x1 = cx + half_width + 1
    roi = np.zeros(spatial_shape, dtype=bool)
    y0c = max(0, y0)
    x0c = max(0, x0)
    y1c = min(spatial_shape[0], y1)
    x1c = min(spatial_shape[1], x1)
    roi[y0c:y1c, x0c:x1c] = True
    return cx, cy, roi, (y0, y1, x0, x1)


def build_roi_preview_from_csvs(csv_files):
    preview = []
    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        colmap = {c.lower().strip(): c for c in df.columns}
        if "bx" not in colmap or "by" not in colmap:
            continue
        bx_col = colmap["bx"]
        by_col = colmap["by"]
        for idx, rec in df.iterrows():
            bx = rec[bx_col]
            by = rec[by_col]
            if pd.isna(bx) or pd.isna(by):
                continue
            preview.append((csv_path.name, int(idx), float(bx), float(by)))
    return preview


def assign_cell_to_roi_from_point(cell_small: np.ndarray, bx, by):
    """
    Assign cell by sampling the reduced 256x256 cell mask at the BX/BY point.
    """
    if pd.isna(bx) or pd.isna(by):
        return 0, "missing_coordinate", 0, np.nan

    x = int(round(float(bx)))
    y = int(round(float(by)))

    if COORDS_ARE_ONE_BASED:
        x -= 1
        y -= 1

    if not (0 <= y < cell_small.shape[0] and 0 <= x < cell_small.shape[1]):
        return 0, "out_of_bounds", 0, np.nan

    cell_id = int(cell_small[y, x])

    if cell_id > 0:
        cell_region_pixel_count = int(np.count_nonzero(cell_small == cell_id))
        return cell_id, "cell_small_point_sample", cell_region_pixel_count, 1.0

    return 0, "unassigned", 0, 0.0


def annotate_cell_ids(ax, label_img, color="white", fontsize=7):
    for lab in np.unique(label_img):
        lab = int(lab)
        if lab <= 0:
            continue
        ys, xs = np.where(label_img == lab)
        if ys.size == 0:
            continue
        cy = float(np.mean(ys))
        cx = float(np.mean(xs))
        ax.text(
            cx,
            cy,
            str(lab),
            color=color,
            fontsize=fontsize,
            ha="center",
            va="center",
            weight="bold",
            alpha=0.95,
        )


def show_cell_diagnostics_with_overlay(
    stack,
    cell_fullres,
    cell_small,
    title,
    preview_rois=None,
    assigned_rois=None,
    channel_index=2,
):
    """
    Left: original cell TIFF
    Middle: reduced 256x256 cell TIFF
    Right: reduced cell TIFF overlaid on NanoSIMS channel with ROI positions
    """
    fig, axes = plt.subplots(1, 3, figsize=(22, 8))
    ax0, ax1, ax2 = axes

    def draw_label_boundaries(ax, label_img, color_map=None):
        labs = [int(v) for v in np.unique(label_img) if int(v) > 0]
        if color_map is None:
            color_map = {lab: ("cyan" if i == 0 else "magenta" if i == 1 else "lime") for i, lab in enumerate(labs)}
        for lab in labs:
            mask = label_img == lab
            if not np.any(mask):
                continue
            ax.contour(mask.astype(float), levels=[0.5], colors=[color_map.get(lab, "white")], linewidths=2.0)
            ys, xs = np.where(mask)
            if ys.size:
                ax.text(
                    float(np.mean(xs)),
                    float(np.mean(ys)),
                    str(lab),
                    color="white",
                    fontsize=10,
                    weight="bold",
                    ha="center",
                    va="center",
                )

    ax0.imshow(np.zeros_like(cell_fullres), cmap="gray", interpolation="nearest")
    draw_label_boundaries(ax0, cell_fullres)
    ax0.set_title(f"Original cell TIFF\nshape={cell_fullres.shape}")
    ax0.set_axis_off()

    ax1.imshow(np.zeros_like(cell_small), cmap="gray", interpolation="nearest")
    draw_label_boundaries(ax1, cell_small)
    ax1.set_title("Reduced cell TIFF (256x256)")
    ax1.set_axis_off()

    ax2.imshow(stack[channel_index], cmap="gray", interpolation="nearest")
    cell_overlay = np.ma.masked_where(cell_small <= 0, cell_small)
    ax2.imshow(cell_overlay, cmap="tab20", alpha=0.35, interpolation="nearest")
    draw_label_boundaries(ax2, cell_small)

    if preview_rois is not None:
        for csv_name, idx, bx, by in preview_rois:
            cx, cy, roi_mask, _ = measure_roi_mask(stack.shape[1:], bx, by, half_width=ROI_HALF_WIDTH)
            ax2.contour(roi_mask.astype(float), levels=[0.5], colors="yellow", linewidths=1.0)
            ax2.scatter([cx], [cy], marker="o", s=18, color="yellow")
            ax2.text(
                cx + 1,
                cy + 1,
                f"{idx}",
                color="yellow",
                fontsize=6,
                weight="bold",
                ha="left",
                va="top",
            )

    if assigned_rois is not None:
        for roi_mask, (cy, cx), lab, cell_id in assigned_rois:
            ax2.contour(roi_mask.astype(float), levels=[0.5], colors="red", linewidths=1.25)
            ax2.scatter([cx], [cy], marker="x", s=40, color="red")
            ax2.text(
                cx + 1,
                cy + 1,
                f"ROI {lab}\nCell {cell_id}",
                color="cyan",
                fontsize=7,
                weight="bold",
                ha="left",
                va="top",
            )

    ax2.set_title(f"NanoSIMS overlay (channel {channel_index + 1})")
    ax2.set_axis_off()

    fig.suptitle(title, y=0.98)
    plt.tight_layout()
    plt.show()


def show_overlay_all_rois_with_cells(
    stack,
    cell_small,
    roi_masks,
    centers,
    labels,
    cell_ids,
    title,
    channel_index=2,
):
    img = stack[channel_index]
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(img, cmap="gray", interpolation="nearest")

    cell_overlay = np.ma.masked_where(cell_small <= 0, cell_small)
    ax.imshow(cell_overlay, cmap="tab20", alpha=0.35, interpolation="nearest")

    for lab in np.unique(cell_small):
        lab = int(lab)
        if lab <= 0:
            continue
        ys, xs = np.where(cell_small == lab)
        if ys.size == 0:
            continue
        cy = float(np.mean(ys))
        cx = float(np.mean(xs))
        ax.text(
            cx,
            cy,
            str(lab),
            color="white",
            fontsize=7,
            ha="center",
            va="center",
            weight="bold",
            alpha=0.9,
        )

    for roi_mask, (cy, cx), label, cell_id in zip(roi_masks, centers, labels, cell_ids):
        ax.contour(roi_mask.astype(float), levels=[0.5], colors="red", linewidths=1.25)
        ax.scatter([cx], [cy], marker="x", s=40, color="red")
        ax.text(
            cx + 1,
            cy + 1,
            f"ROI {label}\nCell {cell_id}",
            color="yellow",
            fontsize=7,
            weight="bold",
            ha="left",
            va="top",
        )

    ax.set_title(title)
    ax.set_axis_off()
    plt.show()


def compute_row_for_roi(
    stack,
    channel_names,
    stack_mean_norm,
    cell_stack_mean_norm,
    cell_summed_mean_norm,
    cell_sum_mean,
    csv_path: Path,
    stack_path: Path,
    cell_path: Path,
    section_name: str,
    label_idx: int,
    roi_mask: np.ndarray,
    eroded_mask: np.ndarray,
    cx: int,
    cy: int,
    roi_bounds,
    cell_label_id: int,
    cell_annotation_method: str,
    cell_region_pixel_count: int,
    roi_cell_fraction: float,
    original_mask_pixel_count: int,
):
    sample_name, area_name = parse_section_folder_name(section_name)
    props = regionprops(roi_mask.astype(np.uint8))
    prop = props[0] if props else None

    major_px = float(prop.major_axis_length) if prop is not None else np.nan
    minor_px = float(prop.minor_axis_length) if prop is not None else np.nan
    perim_px = float(prop.perimeter) if prop is not None else np.nan
    aspect_ratio = float(major_px / minor_px) if (prop is not None and np.isfinite(minor_px) and minor_px > 0) else np.nan

    row = {
        "sample": sample_name,
        "label_id": int(label_idx),
        "source_label_file": f"Sarcomere_{sample_name}_{area_name}",
        "source_stack_file": stack_path.name,
        "source_em_file": DEFAULT_SOURCE_EM_FILE,
        "source_cell_file": cell_path.name,
        "cell_label_id": int(cell_label_id),
        "cell_region_pixel_count": int(cell_region_pixel_count),
        "centroid_in_cell_mask": np.nan,
        "pixel_size_nm": float(PIXEL_SIZE_NM),
        "normalization_method": "whole_field_mean",
        "cell_normalization_method": "cell_small_point_sample" if cell_label_id > 0 else "none",
        "original_label_pixel_count": int(original_mask_pixel_count),
        "strict_reduced_pixel_count": int(np.count_nonzero(roi_mask)),
        "strict_eroded_pixel_count": int(np.count_nonzero(eroded_mask)),
        "area_nm2": float(np.count_nonzero(roi_mask) * PIXEL_SIZE_NM**2),
        "perimeter_nm": float(perim_px * PIXEL_SIZE_NM) if np.isfinite(perim_px) else np.nan,
        "aspect_ratio": aspect_ratio,
        "major_axis_length_px": major_px,
        "minor_axis_length_px": minor_px,
        "major_axis_length_nm": float(major_px * PIXEL_SIZE_NM) if np.isfinite(major_px) else np.nan,
        "minor_axis_length_nm": float(minor_px * PIXEL_SIZE_NM) if np.isfinite(minor_px) else np.nan,
        "centroid_x_px_fullres": float(cx),
        "centroid_y_px_fullres": float(cy),
        "centroid_x_px_reduced": float(cx),
        "centroid_y_px_reduced": float(cy),
        "centroid_in_strict_reduced_mask": bool((0 <= cy < roi_mask.shape[0]) and (0 <= cx < roi_mask.shape[1]) and roi_mask[cy, cx]),
        "cell_12C14N_12C15N_mean": float(cell_sum_mean) if np.isfinite(cell_sum_mean) else np.nan,
        "source_section": section_name,
        "source_tiff_basename": stack_path.stem,
        "roi_cell_fraction": float(roi_cell_fraction),
        "gmm_cluster": np.nan,
        "gmm_cluster_label": DEFAULT_LABEL,
        "_is_mito": False,
        "_is_mitophagophore": False,
        "_is_mitolysosome": False,
        "_is_partial": False,
    }

    for ch, name in enumerate(channel_names):
        strict_raw = stack[ch][roi_mask]
        strict_mean = stack_mean_norm[ch][roi_mask]
        strict_er_raw = stack[ch][eroded_mask] if np.any(eroded_mask) else np.array([])
        strict_er_mean = stack_mean_norm[ch][eroded_mask] if np.any(eroded_mask) else np.array([])

        if cell_stack_mean_norm is not None:
            strict_cell = cell_stack_mean_norm[ch][roi_mask]
            strict_er_cell = cell_stack_mean_norm[ch][eroded_mask] if np.any(eroded_mask) else np.array([])
        else:
            strict_cell = np.array([])
            strict_er_cell = np.array([])

        add_stats(row, f"strict_reduced_raw_{name}", strict_raw)
        add_stats(row, f"strict_reduced_mean_norm_{name}", strict_mean)
        add_stats(row, f"strict_eroded_raw_{name}", strict_er_raw)
        add_stats(row, f"strict_eroded_mean_norm_{name}", strict_er_mean)
        add_stats(row, f"strict_reduced_cell_norm_{name}", strict_cell)
        add_stats(row, f"strict_eroded_cell_norm_{name}", strict_er_cell)

        row[f"centroid_strict_reduced_cell_norm_{name}"] = (
            float(cell_stack_mean_norm[ch][cy, cx])
            if cell_stack_mean_norm is not None and 0 <= cy < cell_stack_mean_norm.shape[1] and 0 <= cx < cell_stack_mean_norm.shape[2]
            else np.nan
        )
        row[f"centroid_strict_eroded_cell_norm_{name}"] = row[f"centroid_strict_reduced_cell_norm_{name}"]

    if "12C14N" in channel_names and "12C15N" in channel_names:
        ch2 = channel_names.index("12C14N")
        ch3 = channel_names.index("12C15N")
        summed = stack[ch2] + stack[ch3]
        frac_15n = np.full_like(summed, np.nan, dtype=np.float64)
        np.divide(stack[ch3], summed, out=frac_15n, where=(summed != 0))
        summed_mean_norm = stack_mean_norm[ch2] + stack_mean_norm[ch3]

        add_stats(row, "strict_reduced_raw_sum_12C14N_12C15N", summed[roi_mask])
        add_stats(row, "strict_eroded_raw_sum_12C14N_12C15N", summed[eroded_mask] if np.any(eroded_mask) else np.array([]))
        add_stats(row, "strict_reduced_mean_norm_sum_12C14N_12C15N", summed_mean_norm[roi_mask])
        add_stats(row, "strict_eroded_mean_norm_sum_12C14N_12C15N", summed_mean_norm[eroded_mask] if np.any(eroded_mask) else np.array([]))
        add_stats(row, "strict_reduced_raw_fractional_15N", frac_15n[roi_mask])
        add_stats(row, "strict_eroded_raw_fractional_15N", frac_15n[eroded_mask] if np.any(eroded_mask) else np.array([]))

        if cell_stack_mean_norm is not None and cell_summed_mean_norm is not None:
            add_stats(row, "strict_reduced_cell_norm_sum_12C14N_12C15N", cell_summed_mean_norm[roi_mask])
            add_stats(row, "strict_eroded_cell_norm_sum_12C14N_12C15N", cell_summed_mean_norm[eroded_mask] if np.any(eroded_mask) else np.array([]))
            row["centroid_strict_reduced_cell_norm_sum_12C14N_12C15N"] = float(cell_summed_mean_norm[cy, cx]) if 0 <= cy < cell_summed_mean_norm.shape[0] and 0 <= cx < cell_summed_mean_norm.shape[1] else np.nan
            row["centroid_strict_eroded_cell_norm_sum_12C14N_12C15N"] = row["centroid_strict_reduced_cell_norm_sum_12C14N_12C15N"]
        else:
            add_stats(row, "strict_reduced_cell_norm_sum_12C14N_12C15N", np.array([]))
            add_stats(row, "strict_eroded_cell_norm_sum_12C14N_12C15N", np.array([]))
            row["centroid_strict_reduced_cell_norm_sum_12C14N_12C15N"] = np.nan
            row["centroid_strict_eroded_cell_norm_sum_12C14N_12C15N"] = np.nan

        row["strict_reduced_cell_norm_sum_12C14N_12C15N_values"] = []
        row["strict_eroded_cell_norm_sum_12C14N_12C15N_values"] = []
        row["centroid_strict_reduced_raw_sum_12C14N_12C15N"] = float(summed[cy, cx]) if 0 <= cy < summed.shape[0] and 0 <= cx < summed.shape[1] else np.nan
        row["centroid_strict_eroded_raw_sum_12C14N_12C15N"] = row["centroid_strict_reduced_raw_sum_12C14N_12C15N"]
        row["centroid_strict_reduced_raw_fractional_15N"] = float(frac_15n[cy, cx]) if 0 <= cy < frac_15n.shape[0] and 0 <= cx < frac_15n.shape[1] else np.nan
        row["centroid_strict_eroded_raw_fractional_15N"] = row["centroid_strict_reduced_raw_fractional_15N"]
        row["cell_12C14N_12C15N_mean"] = float(cell_sum_mean) if np.isfinite(cell_sum_mean) else np.nan

    return row


def process_section_folder(section_dir: Path):
    sample_name, area_name = parse_section_folder_name(section_dir.name)

    csv_dir_candidates = [
        section_dir / "label_sarcomere_manual",
        section_dir / "label_sacromere_manual",
    ]
    csv_dir = next((p for p in csv_dir_candidates if p.exists()), None)
    if csv_dir is None:
        raise FileNotFoundError(
            f"Missing CSV folder in {section_dir}: expected 'label_sarcomere_manual' or 'label_sacromere_manual'"
        )

    stack_dir = section_dir / "nanosims_stack"
    cell_dir = section_dir / "label_cells"

    if not stack_dir.exists():
        raise FileNotFoundError(f"Missing NanoSIMS folder: {stack_dir}")
    if not cell_dir.exists():
        raise FileNotFoundError(f"Missing cell-label folder: {cell_dir}")

    csv_files = find_csv_files(csv_dir)
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {csv_dir}")

    stack_path = find_first_tiff(stack_dir)
    cell_path = find_first_tiff(cell_dir)

    print(f"\n=== Section: {section_dir.name} ===")
    print(f"Using stack: {stack_path.name}")
    print(f"Using cell labels: {cell_path.name}")

    stack = load_stack(stack_path)
    channel_names = infer_channel_names(stack.shape[0])
    stack_mean_norm, field_means = whole_field_mean_norm(stack, channel_names)

    # Load full-res cell label image, then reduce to 256x256 by modal block value
    cell_fullres, cell_small = load_cell_labels(cell_path, target_shape=stack.shape[1:])
    if cell_small.shape != stack.shape[1:]:
        raise ValueError(f"Reduced cell label shape {cell_small.shape} does not match NanoSIMS shape {stack.shape[1:]}")

    # Keep as a channel for visual overlay
    stack_with_cell = append_cell_channel_to_stack(stack, cell_small)

    preview_rois = build_roi_preview_from_csvs(csv_files)

    # Preview overlay before assignment
    show_cell_diagnostics_with_overlay(
        stack=stack_with_cell,
        cell_fullres=cell_fullres,
        cell_small=cell_small,
        title=f"{section_dir.name} | {stack_path.name} | {cell_path.name}",
        preview_rois=preview_rois,
        assigned_rois=None,
        channel_index=DISPLAY_CHANNEL_INDEX,
    )

    rows = []
    overlay_masks = []
    overlay_centers = []
    overlay_labels = []
    overlay_cell_ids = []

    cell_norm_cache = {}

    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        colmap = {c.lower().strip(): c for c in df.columns}
        if "bx" not in colmap or "by" not in colmap:
            raise ValueError(f"{csv_path.name} must contain BX and BY columns")
        bx_col = colmap["bx"]
        by_col = colmap["by"]

        print(f"  Processing CSV: {csv_path.name}")

        for idx, rec in df.iterrows():
            bx = rec[bx_col]
            by = rec[by_col]
            if pd.isna(bx) or pd.isna(by):
                continue

            cx, cy, roi_mask, roi_bounds = measure_roi_mask(
                stack.shape[1:], bx, by, half_width=ROI_HALF_WIDTH
            )
            original_mask_pixel_count = int(np.count_nonzero(roi_mask))
            eroded_mask = binary_erosion(
                roi_mask,
                footprint=np.ones((2 * 1 + 1, 2 * 1 + 1), dtype=bool)
            )

            # NEW BEHAVIOR:
            # Assign cell based on the reduced 256x256 mask at the BX/BY point
            cell_label_id, cell_method, cell_region_pixel_count, roi_cell_fraction = assign_cell_to_roi_from_point(
                cell_small=cell_small,
                bx=bx,
                by=by,
            )

            if cell_label_id == 0:
                continue

            if cell_label_id not in cell_norm_cache:
                cell_region_mask = cell_small == cell_label_id
                if np.any(cell_region_mask):
                    cell_stack_mean_norm, cell_field_means = whole_field_mean_norm_masked(stack, channel_names, cell_region_mask)
                    ch2 = channel_names.index("12C14N")
                    ch3 = channel_names.index("12C15N")
                    cell_summed = stack[ch2] + stack[ch3]
                    cell_sum_vals = cell_summed[cell_region_mask]
                    cell_sum_mean = float(np.mean(cell_sum_vals)) if cell_sum_vals.size else 1.0
                    if not np.isfinite(cell_sum_mean) or cell_sum_mean == 0:
                        cell_sum_mean = 1.0
                    cell_summed_mean_norm = cell_summed / cell_sum_mean
                    cell_norm_cache[cell_label_id] = (cell_stack_mean_norm, cell_field_means, cell_summed_mean_norm, cell_sum_mean)
                else:
                    cell_norm_cache[cell_label_id] = (None, None, None, np.nan)

            if cell_label_id > 0:
                cell_stack_mean_norm, cell_field_means, cell_summed_mean_norm, cell_sum_mean = cell_norm_cache[cell_label_id]
            else:
                cell_stack_mean_norm, cell_field_means, cell_summed_mean_norm, cell_sum_mean = None, None, None, np.nan

            overlay_masks.append(roi_mask)
            overlay_centers.append((cy, cx))
            overlay_labels.append(f"{csv_path.stem}:{idx}")
            overlay_cell_ids.append(int(cell_label_id))

            row = compute_row_for_roi(
                stack=stack,
                channel_names=channel_names,
                stack_mean_norm=stack_mean_norm,
                cell_stack_mean_norm=cell_stack_mean_norm,
                cell_summed_mean_norm=cell_summed_mean_norm,
                cell_sum_mean=cell_sum_mean,
                csv_path=csv_path,
                stack_path=stack_path,
                cell_path=cell_path,
                section_name=section_dir.name,
                label_idx=int(idx) + 1,
                roi_mask=roi_mask,
                eroded_mask=eroded_mask,
                cx=cx,
                cy=cy,
                roi_bounds=roi_bounds,
                cell_label_id=int(cell_label_id),
                cell_annotation_method=cell_method,
                cell_region_pixel_count=int(cell_region_pixel_count),
                roi_cell_fraction=roi_cell_fraction,
                original_mask_pixel_count=original_mask_pixel_count,
            )

            row["sample"] = sample_name
            row["area"] = area_name
            rows.append(row)

    if overlay_masks:
        show_cell_diagnostics_with_overlay(
            stack=stack_with_cell,
            cell_fullres=cell_fullres,
            cell_small=cell_small,
            title=f"{section_dir.name} | assigned ROIs",
            preview_rois=None,
            assigned_rois=list(zip(overlay_masks, overlay_centers, overlay_labels, overlay_cell_ids)),
            channel_index=DISPLAY_CHANNEL_INDEX,
        )

        show_overlay_all_rois_with_cells(
            stack=stack_with_cell,
            cell_small=cell_small,
            roi_masks=overlay_masks,
            centers=overlay_centers,
            labels=overlay_labels,
            cell_ids=overlay_cell_ids,
            title=f"{section_dir.name} | {stack_path.name}",
            channel_index=DISPLAY_CHANNEL_INDEX,
        )

    section_df = pd.DataFrame(rows)
    section_df = section_df.reindex(columns=MORPH_COLS)

    section_out = OUTPUT_ROOT / f"{section_dir.name}_sarcomere_measurements.csv"
    section_df.to_csv(section_out, index=False)
    print(f"Saved section CSV: {section_out.resolve()}")

    return section_df

# -------------------------------------------------------------
# Run all sections
# -------------------------------------------------------------
section_dirs = sorted([p for p in INPUT_ROOT.iterdir() if p.is_dir()])
if not section_dirs:
    raise FileNotFoundError(f"No section folders found in {INPUT_ROOT.resolve()}")

all_dfs = []
for section_dir in section_dirs:
    try:
        if not re.search(r"(?i)S0*\d+", section_dir.name) or not re.search(r"(?i)Ar0*\d+", section_dir.name):
            print(f"Skipping {section_dir.name}: does not look like a section folder")
            continue

        section_df = process_section_folder(section_dir)
        if not section_df.empty:
            all_dfs.append(section_df)

    except Exception as e:
        print(f"Skipping {section_dir.name}: {e}")

if not all_dfs:
    raise RuntimeError("No measurements were produced.")

final_df = pd.concat(all_dfs, ignore_index=True)
final_df = final_df.reindex(columns=MORPH_COLS)

# Save sarcomere-only schema-matched output
sarcomere_schema_out = OUTPUT_ROOT / "combined_manual_sarcomere_morphology_schema.csv"
final_df.to_csv(sarcomere_schema_out, index=False)
print(f"\nSaved sarcomere morphology-schema CSV to: {sarcomere_schema_out.resolve()}")

display(final_df.head())

# -------------------------------------------------------------
# Create addended version of the existing combined morphology file
# -------------------------------------------------------------
base_morph_df = pd.read_csv(BASE_MORPHOLOGY_PATH)
base_morph_df = base_morph_df.reindex(columns=MORPH_COLS)

sarcomere_for_append = final_df.reindex(columns=base_morph_df.columns)
addended_df = pd.concat([base_morph_df, sarcomere_for_append], ignore_index=True)

addended_out = OUTPUT_ROOT / "combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated_addended.csv"
addended_df.to_csv(addended_out, index=False)
print(f"Saved addended morphology CSV to: {addended_out.resolve()}")

addended_df

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import tifffile as tiff
import matplotlib.pyplot as plt
from IPython.display import display
from skimage.measure import regionprops, label as cc_label
from skimage.morphology import binary_erosion, remove_small_holes

# =============================================================
# Cell measurements based on the reduced 256x256 cell mask
#
# This cell:
# - reads the addended CSV from the prior cell
# - measures each unique cell label in the reduced 256x256 cell mask
# - appends those cell rows in the same morphology-style schema
# - writes a combined CSV with Sarcomeres + Cells
# =============================================================

# -------------------------------------------------------------
# Paths
# -------------------------------------------------------------
INPUT_ROOT = Path("input")

INPUT_CSV = Path(
    r"output_manual_sarcomere_measurements\combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated_addended.csv"
)

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Could not find input CSV:\n{INPUT_CSV.resolve()}")

OUTPUT_ROOT = Path("output_manual_sarcomere_measurements")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BASE_MORPHOLOGY_CANDIDATES = [
    Path("combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated.csv"),
    Path("/mnt/data/combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated.csv"),
]

BASE_MORPHOLOGY_PATH = None
for p in BASE_MORPHOLOGY_CANDIDATES:
    if p.exists():
        BASE_MORPHOLOGY_PATH = p
        break

if BASE_MORPHOLOGY_PATH is None:
    raise FileNotFoundError(
        "Could not find combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated.csv"
    )

print("INPUT_ROOT :", INPUT_ROOT.resolve())
print("INPUT_CSV  :", INPUT_CSV.resolve())
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())
print("BASE_MORPHOLOGY_PATH:", BASE_MORPHOLOGY_PATH.resolve())

# -------------------------------------------------------------
# Settings
# -------------------------------------------------------------
TARGET_H, TARGET_W = 256, 256
COORDS_ARE_ONE_BASED = False
DISPLAY_CHANNEL_INDEX = 2
PIXEL_SIZE_NM = 1.751

DEFAULT_SOURCE_EM_FILE = ""

CELL_FLIPUD = False
CELL_FLIPLR = False
CELL_ROTATE_K = 0

EXPECTED_CHANNELS_7 = ["16O", "12C21H", "12C14N", "12C15N", "29Si", "31P", "32S"]
EXPECTED_CHANNELS_6 = ["16O", "12C21H", "12C14N", "12C15N", "31P", "32S"]
MORPH_COLS = pd.read_csv(BASE_MORPHOLOGY_PATH, nrows=0).columns.tolist()

# -------------------------------------------------------------
# Helpers
# -------------------------------------------------------------
def infer_channel_names(n_channels):
    if n_channels == 7:
        return EXPECTED_CHANNELS_7.copy()
    if n_channels == 6:
        return EXPECTED_CHANNELS_6.copy()
    raise ValueError(f"Unsupported channel count: {n_channels}. Expected 6 or 7.")


def finite_values(a):
    a = np.asarray(a, dtype=float).ravel()
    return a[np.isfinite(a)]


def summarize(values):
    v = finite_values(values)
    if v.size == 0:
        return {
            "n": 0, "mean": np.nan, "median": np.nan, "std": np.nan,
            "min": np.nan, "max": np.nan, "q1": np.nan, "q3": np.nan, "iqr": np.nan
        }
    q1 = float(np.percentile(v, 25))
    q3 = float(np.percentile(v, 75))
    return {
        "n": int(v.size),
        "mean": float(np.mean(v)),
        "median": float(np.median(v)),
        "std": float(np.std(v)),
        "min": float(np.min(v)),
        "max": float(np.max(v)),
        "q1": q1,
        "q3": q3,
        "iqr": float(q3 - q1),
    }


def add_stats(row, prefix, values):
    s = summarize(values)
    for k, v in s.items():
        row[f"{prefix}_{k}"] = v


def whole_field_mean_norm(stack, channel_names):
    n_channels = stack.shape[0]
    if len(channel_names) != n_channels:
        raise ValueError(f"Channel-name count ({len(channel_names)}) does not match stack channels ({n_channels}).")
    means = np.empty(n_channels, dtype=float)
    for ch in range(n_channels):
        vals = stack[ch]
        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size == 0:
            means[ch] = 1.0
        else:
            mu = float(np.mean(finite_vals))
            means[ch] = mu if np.isfinite(mu) and mu != 0 else 1.0
    return stack / means[:, None, None], means


def whole_field_mean_norm_masked(stack, channel_names, mask):
    n_channels = stack.shape[0]
    if len(channel_names) != n_channels:
        raise ValueError(f"Channel-name count ({len(channel_names)}) does not match stack channels ({n_channels}).")
    mask = np.asarray(mask, dtype=bool)
    if mask.shape != stack.shape[1:]:
        raise ValueError(f"Mask shape {mask.shape} does not match stack spatial shape {stack.shape[1:]}.")
    means = np.empty(n_channels, dtype=float)
    for ch in range(n_channels):
        vals = stack[ch][mask]
        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size == 0:
            means[ch] = 1.0
        else:
            mu = float(np.mean(finite_vals))
            means[ch] = mu if np.isfinite(mu) and mu != 0 else 1.0
    return stack / means[:, None, None], means


def find_first_tiff(folder: Path):
    files = sorted(list(folder.glob("*.tif")) + list(folder.glob("*.tiff")))
    if not files:
        raise FileNotFoundError(f"No TIFF files found in {folder}")
    if len(files) > 1:
        print(f"Warning: multiple TIFFs found in {folder}, using {files[0].name}")
    return files[0]


def load_stack(stack_path: Path):
    stack = tiff.imread(str(stack_path))
    if stack.ndim != 3:
        raise ValueError(f"Expected a 3D TIFF stack, got shape {stack.shape} for {stack_path.name}")
    if stack.shape[1:] == (256, 256):
        pass
    elif stack.shape[:2] == (256, 256):
        stack = np.moveaxis(stack, -1, 0)
    else:
        raise ValueError(f"Expected 256x256 spatial shape, got {stack.shape} for {stack_path.name}")
    if stack.shape[0] not in (6, 7):
        raise ValueError(f"Expected 6 or 7 channels, got {stack.shape[0]} in {stack_path.name}")
    return stack.astype(np.float64)


def parse_section_folder_name(section_name: str):
    s = str(section_name)
    m_sample = re.search(r"(?i)S0*(\d+)", s)
    m_area = re.search(r"(?i)Ar\s*0*(\d+)", s)
    if m_sample is None or m_area is None:
        raise ValueError(f"Could not parse sample/area from section folder name '{section_name}'.")
    sample = f"S{int(m_sample.group(1))}"
    area = f"Ar{int(m_area.group(1))}"
    return sample, area


def clean_cell_labels(cell_img: np.ndarray):
    cell_img = np.rint(cell_img).astype(np.int32)
    cleaned = np.zeros_like(cell_img, dtype=np.int32)

    for lab in np.unique(cell_img):
        lab = int(lab)
        if lab <= 0:
            continue

        mask = cell_img == lab
        if not np.any(mask):
            continue

        mask = remove_small_holes(mask, area_threshold=mask.size)
        cc = cc_label(mask, connectivity=2)
        if cc.max() == 0:
            continue

        counts = np.bincount(cc.ravel())
        counts[0] = 0
        keep = int(np.argmax(counts))
        cleaned[cc == keep] = lab

    return cleaned


def block_mode(values):
    values = np.asarray(values).ravel().astype(np.int32)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0
    positive = values[values > 0]
    if positive.size > 0:
        values = positive
    vals, cnts = np.unique(values, return_counts=True)
    return int(vals[np.argmax(cnts)])


def downsample_label_image_to_target(label_img: np.ndarray, target_shape=(256, 256)):
    if label_img.ndim != 2:
        raise ValueError(f"Expected 2D label image, got shape {label_img.shape}")

    src_h, src_w = label_img.shape
    tgt_h, tgt_w = target_shape

    y_edges = np.linspace(0, src_h, tgt_h + 1)
    x_edges = np.linspace(0, src_w, tgt_w + 1)

    out = np.zeros((tgt_h, tgt_w), dtype=np.int32)

    for i in range(tgt_h):
        y0 = int(np.floor(y_edges[i]))
        y1 = int(np.ceil(y_edges[i + 1]))
        y1 = min(max(y1, y0 + 1), src_h)

        for j in range(tgt_w):
            x0 = int(np.floor(x_edges[j]))
            x1 = int(np.ceil(x_edges[j + 1]))
            x1 = min(max(x1, x0 + 1), src_w)

            block = label_img[y0:y1, x0:x1]
            out[i, j] = block_mode(block)

    return out


def load_cell_labels(cell_path: Path, target_shape=(256, 256)):
    cell_img = tiff.imread(str(cell_path))
    if cell_img.ndim != 2:
        raise ValueError(f"Expected a 2D cell label TIFF, got shape {cell_img.shape} for {cell_path.name}")

    cell_img = np.rint(cell_img).astype(np.int32)

    if CELL_ROTATE_K:
        cell_img = np.rot90(cell_img, k=CELL_ROTATE_K)
    if CELL_FLIPUD:
        cell_img = np.flipud(cell_img)
    if CELL_FLIPLR:
        cell_img = np.fliplr(cell_img)

    cell_small = downsample_label_image_to_target(cell_img, target_shape=target_shape)
    return cell_img, cell_small


def infer_field_id_from_row(row):
    candidates = [
        row.get("source_label_file", ""),
        row.get("source_stack_file", ""),
        row.get("source_cell_file", ""),
        row.get("sample", ""),
        row.get("source_section", ""),
    ]
    for value in candidates:
        s = str(value)
        m = re.search(r"(S\d+_Ar\d+)", s, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    return None


def assign_cell_from_centroid(cell_small: np.ndarray, x, y):
    if pd.isna(x) or pd.isna(y):
        return 0, "missing_coordinate", 0

    xx = int(round(float(x)))
    yy = int(round(float(y)))

    if COORDS_ARE_ONE_BASED:
        xx -= 1
        yy -= 1

    if not (0 <= yy < cell_small.shape[0] and 0 <= xx < cell_small.shape[1]):
        return 0, "out_of_bounds", 0

    cell_id = int(cell_small[yy, xx])
    if cell_id > 0:
        pixel_count = int(np.count_nonzero(cell_small == cell_id))
        return cell_id, "cell_small_centroid_sample", pixel_count

    return 0, "unassigned", 0


def infer_object_type(s):
    s = str(s).lower()
    if "sarcomere" in s:
        return "Sarcomere"
    if "mitolysosome" in s:
        return "Mitolysosome"
    if "mitophagophore" in s:
        return "Mitophagophore"
    if "mitochond" in s or "mito" in s:
        return "Mitochondria"
    return "Other"


def compute_row_for_cell(
    stack,
    channel_names,
    stack_mean_norm,
    cell_stack_mean_norm,
    cell_summed_mean_norm,
    cell_sum_mean,
    stack_path: Path,
    cell_path: Path,
    section_name: str,
    cell_label_id: int,
    cell_mask: np.ndarray,
):
    sample_name, area_name = parse_section_folder_name(section_name)
    props = regionprops(cell_mask.astype(np.uint8))
    prop = props[0] if props else None

    if prop is not None:
        cy = float(prop.centroid[0])
        cx = float(prop.centroid[1])
        major_px = float(prop.major_axis_length)
        minor_px = float(prop.minor_axis_length)
        perim_px = float(prop.perimeter)
        aspect_ratio = float(major_px / minor_px) if np.isfinite(minor_px) and minor_px > 0 else np.nan
    else:
        cx = cy = major_px = minor_px = perim_px = aspect_ratio = np.nan

    original_mask_pixel_count = int(np.count_nonzero(cell_mask))
    eroded_mask = binary_erosion(
        cell_mask,
        footprint=np.ones((3, 3), dtype=bool)
    )

    row = {
        "sample": sample_name,
        "label_id": int(cell_label_id),
        "source_label_file": f"Cell_{sample_name}_{area_name}",
        "source_stack_file": stack_path.name,
        "source_em_file": DEFAULT_SOURCE_EM_FILE,
        "source_cell_file": cell_path.name,
        "cell_label_id": int(cell_label_id),
        "cell_region_pixel_count": int(original_mask_pixel_count),
        "centroid_in_cell_mask": True,
        "pixel_size_nm": float(PIXEL_SIZE_NM),
        "normalization_method": "whole_field_mean",
        "cell_normalization_method": "cell_small_region",
        "original_label_pixel_count": int(original_mask_pixel_count),
        "strict_reduced_pixel_count": int(np.count_nonzero(cell_mask)),
        "strict_eroded_pixel_count": int(np.count_nonzero(eroded_mask)),
        "area_nm2": float(original_mask_pixel_count * PIXEL_SIZE_NM**2),
        "perimeter_nm": float(perim_px * PIXEL_SIZE_NM) if np.isfinite(perim_px) else np.nan,
        "aspect_ratio": aspect_ratio,
        "major_axis_length_px": major_px,
        "minor_axis_length_px": minor_px,
        "major_axis_length_nm": float(major_px * PIXEL_SIZE_NM) if np.isfinite(major_px) else np.nan,
        "minor_axis_length_nm": float(minor_px * PIXEL_SIZE_NM) if np.isfinite(minor_px) else np.nan,
        "centroid_x_px_fullres": float(cx),
        "centroid_y_px_fullres": float(cy),
        "centroid_x_px_reduced": float(cx),
        "centroid_y_px_reduced": float(cy),
        "centroid_in_strict_reduced_mask": bool(
            np.isfinite(cx) and np.isfinite(cy) and
            0 <= int(round(cy)) < cell_mask.shape[0] and
            0 <= int(round(cx)) < cell_mask.shape[1] and
            cell_mask[int(round(cy)), int(round(cx))]
        ),
        "cell_12C14N_12C15N_mean": float(cell_sum_mean) if np.isfinite(cell_sum_mean) else np.nan,
        "source_section": section_name,
        "source_tiff_basename": stack_path.stem,
        "roi_cell_fraction": 1.0,
        "gmm_cluster": np.nan,
        "gmm_cluster_label": "Cell",
        "_is_mito": False,
        "_is_mitophagophore": False,
        "_is_mitolysosome": False,
        "_is_partial": False,
    }

    for ch, name in enumerate(channel_names):
        strict_raw = stack[ch][cell_mask]
        strict_mean = stack_mean_norm[ch][cell_mask]
        strict_er_raw = stack[ch][eroded_mask] if np.any(eroded_mask) else np.array([])
        strict_er_mean = stack_mean_norm[ch][eroded_mask] if np.any(eroded_mask) else np.array([])

        if cell_stack_mean_norm is not None:
            strict_cell = cell_stack_mean_norm[ch][cell_mask]
            strict_er_cell = cell_stack_mean_norm[ch][eroded_mask] if np.any(eroded_mask) else np.array([])
        else:
            strict_cell = np.array([])
            strict_er_cell = np.array([])

        add_stats(row, f"strict_reduced_raw_{name}", strict_raw)
        add_stats(row, f"strict_reduced_mean_norm_{name}", strict_mean)
        add_stats(row, f"strict_eroded_raw_{name}", strict_er_raw)
        add_stats(row, f"strict_eroded_mean_norm_{name}", strict_er_mean)
        add_stats(row, f"strict_reduced_cell_norm_{name}", strict_cell)
        add_stats(row, f"strict_eroded_cell_norm_{name}", strict_er_cell)

        if cell_stack_mean_norm is not None and np.isfinite(cx) and np.isfinite(cy):
            yy = int(round(cy))
            xx = int(round(cx))
            if 0 <= yy < cell_stack_mean_norm.shape[1] and 0 <= xx < cell_stack_mean_norm.shape[2]:
                row[f"centroid_strict_reduced_cell_norm_{name}"] = float(cell_stack_mean_norm[ch][yy, xx])
                row[f"centroid_strict_eroded_cell_norm_{name}"] = row[f"centroid_strict_reduced_cell_norm_{name}"]
            else:
                row[f"centroid_strict_reduced_cell_norm_{name}"] = np.nan
                row[f"centroid_strict_eroded_cell_norm_{name}"] = np.nan
        else:
            row[f"centroid_strict_reduced_cell_norm_{name}"] = np.nan
            row[f"centroid_strict_eroded_cell_norm_{name}"] = np.nan

    if "12C14N" in channel_names and "12C15N" in channel_names:
        ch2 = channel_names.index("12C14N")
        ch3 = channel_names.index("12C15N")
        summed = stack[ch2] + stack[ch3]
        frac_15n = np.full_like(summed, np.nan, dtype=np.float64)
        np.divide(stack[ch3], summed, out=frac_15n, where=(summed != 0))
        summed_mean_norm = stack_mean_norm[ch2] + stack_mean_norm[ch3]

        add_stats(row, "strict_reduced_raw_sum_12C14N_12C15N", summed[cell_mask])
        add_stats(row, "strict_eroded_raw_sum_12C14N_12C15N", summed[eroded_mask] if np.any(eroded_mask) else np.array([]))
        add_stats(row, "strict_reduced_mean_norm_sum_12C14N_12C15N", summed_mean_norm[cell_mask])
        add_stats(row, "strict_eroded_mean_norm_sum_12C14N_12C15N", summed_mean_norm[eroded_mask] if np.any(eroded_mask) else np.array([]))
        add_stats(row, "strict_reduced_raw_fractional_15N", frac_15n[cell_mask])
        add_stats(row, "strict_eroded_raw_fractional_15N", frac_15n[eroded_mask] if np.any(eroded_mask) else np.array([]))

        if cell_stack_mean_norm is not None and cell_summed_mean_norm is not None:
            add_stats(row, "strict_reduced_cell_norm_sum_12C14N_12C15N", cell_summed_mean_norm[cell_mask])
            add_stats(row, "strict_eroded_cell_norm_sum_12C14N_12C15N", cell_summed_mean_norm[eroded_mask] if np.any(eroded_mask) else np.array([]))
            if np.isfinite(cx) and np.isfinite(cy):
                yy = int(round(cy))
                xx = int(round(cx))
                if 0 <= yy < cell_summed_mean_norm.shape[0] and 0 <= xx < cell_summed_mean_norm.shape[1]:
                    row["centroid_strict_reduced_cell_norm_sum_12C14N_12C15N"] = float(cell_summed_mean_norm[yy, xx])
                    row["centroid_strict_eroded_cell_norm_sum_12C14N_12C15N"] = row["centroid_strict_reduced_cell_norm_sum_12C14N_12C15N"]
                else:
                    row["centroid_strict_reduced_cell_norm_sum_12C14N_12C15N"] = np.nan
                    row["centroid_strict_eroded_cell_norm_sum_12C14N_12C15N"] = np.nan
            else:
                row["centroid_strict_reduced_cell_norm_sum_12C14N_12C15N"] = np.nan
                row["centroid_strict_eroded_cell_norm_sum_12C14N_12C15N"] = np.nan
        else:
            row["centroid_strict_reduced_cell_norm_sum_12C14N_12C15N"] = np.nan
            row["centroid_strict_eroded_cell_norm_sum_12C14N_12C15N"] = np.nan

        row["strict_reduced_cell_norm_sum_12C14N_12C15N_values"] = []
        row["strict_eroded_cell_norm_sum_12C14N_12C15N_values"] = []
        row["centroid_strict_reduced_raw_sum_12C14N_12C15N"] = float(summed[int(round(cy)), int(round(cx))]) if np.isfinite(cx) and np.isfinite(cy) and 0 <= int(round(cy)) < summed.shape[0] and 0 <= int(round(cx)) < summed.shape[1] else np.nan
        row["centroid_strict_eroded_raw_sum_12C14N_12C15N"] = row["centroid_strict_reduced_raw_sum_12C14N_12C15N"]
        row["centroid_strict_reduced_raw_fractional_15N"] = float(frac_15n[int(round(cy)), int(round(cx))]) if np.isfinite(cx) and np.isfinite(cy) and 0 <= int(round(cy)) < frac_15n.shape[0] and 0 <= int(round(cx)) < frac_15n.shape[1] else np.nan
        row["centroid_strict_eroded_raw_fractional_15N"] = row["centroid_strict_reduced_raw_fractional_15N"]
        row["cell_12C14N_12C15N_mean"] = float(cell_sum_mean) if np.isfinite(cell_sum_mean) else np.nan

    return row


def process_cell_section_folder(section_dir: Path):
    sample_name, area_name = parse_section_folder_name(section_dir.name)
    sample_value = f"{sample_name}_{area_name}"

    stack_dir = section_dir / "nanosims_stack"
    cell_dir = section_dir / "label_cells"

    if not stack_dir.exists():
        raise FileNotFoundError(f"Missing NanoSIMS folder: {stack_dir}")
    if not cell_dir.exists():
        raise FileNotFoundError(f"Missing cell-label folder: {cell_dir}")

    stack_path = find_first_tiff(stack_dir)
    cell_path = find_first_tiff(cell_dir)

    stack = load_stack(stack_path)
    channel_names = infer_channel_names(stack.shape[0])
    stack_mean_norm, _ = whole_field_mean_norm(stack, channel_names)

    cell_fullres, cell_small = load_cell_labels(cell_path, target_shape=stack.shape[1:])
    if cell_small.shape != stack.shape[1:]:
        raise ValueError(f"Reduced cell label shape {cell_small.shape} does not match NanoSIMS shape {stack.shape[1:]}")

    rows = []
    unique_cells = [int(v) for v in np.unique(cell_small) if int(v) > 0]

    for cell_label_id in unique_cells:
        cell_mask = cell_small == cell_label_id
        if not np.any(cell_mask):
            continue

        cell_stack_mean_norm, _ = whole_field_mean_norm_masked(stack, channel_names, cell_mask)

        if "12C14N" in channel_names and "12C15N" in channel_names:
            ch2 = channel_names.index("12C14N")
            ch3 = channel_names.index("12C15N")
            cell_summed = stack[ch2] + stack[ch3]
            cell_sum_vals = cell_summed[cell_mask]
            cell_sum_mean = float(np.mean(cell_sum_vals)) if cell_sum_vals.size else 1.0
            if not np.isfinite(cell_sum_mean) or cell_sum_mean == 0:
                cell_sum_mean = 1.0
            cell_summed_mean_norm = cell_summed / cell_sum_mean
        else:
            cell_sum_mean = np.nan
            cell_summed_mean_norm = None

        row = compute_row_for_cell(
            stack=stack,
            channel_names=channel_names,
            stack_mean_norm=stack_mean_norm,
            cell_stack_mean_norm=cell_stack_mean_norm,
            cell_summed_mean_norm=cell_summed_mean_norm,
            cell_sum_mean=cell_sum_mean,
            stack_path=stack_path,
            cell_path=cell_path,
            section_name=section_dir.name,
            cell_label_id=cell_label_id,
            cell_mask=cell_mask,
        )

        row["sample"] = sample_value
        row["area"] = area_name
        rows.append(row)

    return pd.DataFrame(rows)


def read_addended_csv_with_cleanup(path: Path):
    """
    The addended CSV can contain malformed continuation rows.
    This repairs the field_id by carrying forward/backward the nearest valid one.
    """
    df = pd.read_csv(path, low_memory=False)

    if "cell_label_id" not in df.columns:
        raise ValueError("Expected cell_label_id in the addended CSV.")

    df["field_id_raw"] = df.apply(infer_field_id_from_row, axis=1)
    df["field_id"] = df["field_id_raw"].ffill().bfill()

    return df


# -------------------------------------------------------------
# Read the addended CSV and measure cells
# -------------------------------------------------------------
addended_df = read_addended_csv_with_cleanup(INPUT_CSV)

section_dirs = sorted([p for p in INPUT_ROOT.iterdir() if p.is_dir()])
cell_dfs = []

for section_dir in section_dirs:
    try:
        if not re.search(r"(?i)S0*\d+", section_dir.name) or not re.search(r"(?i)Ar0*\d+", section_dir.name):
            continue

        cell_df = process_cell_section_folder(section_dir)
        if not cell_df.empty:
            cell_dfs.append(cell_df)

        print(f"Measured cells for {section_dir.name}: {len(cell_df)} rows")

    except Exception as e:
        print(f"Skipping cell measurement for {section_dir.name}: {e}")

if not cell_dfs:
    raise RuntimeError("No cell measurements were produced.")

cell_final_df = pd.concat(cell_dfs, ignore_index=True)
cell_final_df = cell_final_df.reindex(columns=MORPH_COLS)

cell_out = OUTPUT_ROOT / "combined_manual_cell_morphology_schema.csv"
cell_final_df.to_csv(cell_out, index=False)
print(f"\nSaved cell morphology-schema CSV to: {cell_out.resolve()}")

# Append cell rows to the addended dataframe
cell_rows_for_append = cell_final_df.reindex(columns=addended_df.columns)
addended_with_cells_df = pd.concat([addended_df, cell_rows_for_append], ignore_index=True)

addended_with_cells_out = OUTPUT_ROOT / "combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated_addended_with_cells.csv"
addended_with_cells_df.to_csv(addended_with_cells_out, index=False)
print(f"Saved combined sarcomere+cell CSV to: {addended_with_cells_out.resolve()}")

display(cell_final_df.head())
display(addended_with_cells_df.tail())

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import tifffile as tiff
from IPython.display import display

# =============================================================
# Correct cell_label_id for ALL rows in the addended morphology CSV
# using the working Sarcomere behavior:
#
# full-resolution cell TIFF
# -> 256x256 modal block reduction
# -> point sample at centroid location
#
# This version is robust to a few malformed continuation rows by
# carrying field_id forward/backward from neighboring valid rows.
# =============================================================

# -------------------------------------------------------------
# Paths
# -------------------------------------------------------------
INPUT_ROOT = Path("input")

INPUT_CSV = Path(
    r"output_manual_sarcomere_measurements\combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated_addended_with_cells.csv"
)

# Overwrite in place, as requested.
OUTPUT_CSV = Path(
    r"output_manual_sarcomere_measurements\combined_output_morphology_rated_manually_reviewed_gmm_31P32S_annotated_addended_with_cells.csv"
)

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Could not find input CSV:\n{INPUT_CSV.resolve()}")

print("INPUT_ROOT :", INPUT_ROOT.resolve())
print("INPUT_CSV  :", INPUT_CSV.resolve())
print("OUTPUT_CSV :", OUTPUT_CSV.resolve())

# -------------------------------------------------------------
# Settings
# -------------------------------------------------------------
TARGET_H, TARGET_W = 256, 256
CENTROIDS_ARE_ONE_BASED = False

CELL_FLIPUD = False
CELL_FLIPLR = False
CELL_ROTATE_K = 0

# -------------------------------------------------------------
# Helpers
# -------------------------------------------------------------
def infer_field_id_from_row(row):
    """
    Extract S##_Ar# from any useful source column.
    Returns None if the row is malformed.
    """
    candidates = [
        row.get("source_label_file", ""),
        row.get("source_stack_file", ""),
        row.get("source_cell_file", ""),
        row.get("sample", ""),
        row.get("source_section", ""),
    ]

    for value in candidates:
        s = str(value)
        m = re.search(r"(S\d+_Ar\d+)", s, flags=re.IGNORECASE)
        if m:
            return m.group(1)

    return None


def find_cell_tiff(field_id: str):
    """
    Find Cell_S##_Ar#.tif under input/.
    """
    expected = f"Cell_{field_id}.tif"

    direct = INPUT_ROOT / field_id / "label_cells" / expected
    if direct.exists():
        return direct

    matches = list(INPUT_ROOT.rglob(expected))
    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find {expected} under {INPUT_ROOT.resolve()}")

    if len(matches) > 1:
        print(f"Warning: multiple matches for {expected}; using {matches[0]}")

    return matches[0]


def block_mode(values):
    """
    Modal value in a block.
    Prefer positive labels over 0.
    """
    values = np.asarray(values).ravel().astype(np.int32)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return 0

    positive = values[values > 0]
    if positive.size > 0:
        values = positive

    vals, cnts = np.unique(values, return_counts=True)
    return int(vals[np.argmax(cnts)])


def downsample_label_image_to_target(label_img: np.ndarray, target_shape=(256, 256)):
    """
    Reduce a labeled cell image to target_shape by taking the modal label
    within each block.
    """
    if label_img.ndim != 2:
        raise ValueError(f"Expected 2D label image, got shape {label_img.shape}")

    src_h, src_w = label_img.shape
    tgt_h, tgt_w = target_shape

    y_edges = np.linspace(0, src_h, tgt_h + 1)
    x_edges = np.linspace(0, src_w, tgt_w + 1)

    out = np.zeros((tgt_h, tgt_w), dtype=np.int32)

    for i in range(tgt_h):
        y0 = int(np.floor(y_edges[i]))
        y1 = int(np.ceil(y_edges[i + 1]))
        y1 = min(max(y1, y0 + 1), src_h)

        for j in range(tgt_w):
            x0 = int(np.floor(x_edges[j]))
            x1 = int(np.ceil(x_edges[j + 1]))
            x1 = min(max(x1, x0 + 1), src_w)

            block = label_img[y0:y1, x0:x1]
            out[i, j] = block_mode(block)

    return out


def load_cell_labels(cell_path: Path, target_shape=(256, 256)):
    """
    Load full-resolution cell TIFF and reduce to 256x256.
    """
    cell_img = tiff.imread(str(cell_path))
    if cell_img.ndim != 2:
        raise ValueError(f"Expected a 2D cell label TIFF, got shape {cell_img.shape} for {cell_path.name}")

    cell_img = np.rint(cell_img).astype(np.int32)

    if CELL_ROTATE_K:
        cell_img = np.rot90(cell_img, k=CELL_ROTATE_K)
    if CELL_FLIPUD:
        cell_img = np.flipud(cell_img)
    if CELL_FLIPLR:
        cell_img = np.fliplr(cell_img)

    cell_small = downsample_label_image_to_target(cell_img, target_shape=target_shape)
    return cell_img, cell_small


def assign_cell_from_centroid(cell_small: np.ndarray, x, y):
    """
    Assign cell by point sampling the reduced 256x256 cell mask.
    """
    if pd.isna(x) or pd.isna(y):
        return 0, "missing_coordinate", 0

    xx = int(round(float(x)))
    yy = int(round(float(y)))

    if CENTROIDS_ARE_ONE_BASED:
        xx -= 1
        yy -= 1

    if not (0 <= yy < cell_small.shape[0] and 0 <= xx < cell_small.shape[1]):
        return 0, "out_of_bounds", 0

    cell_id = int(cell_small[yy, xx])

    if cell_id > 0:
        pixel_count = int(np.count_nonzero(cell_small == cell_id))
        return cell_id, "cell_small_centroid_sample", pixel_count

    return 0, "unassigned", 0


def infer_object_type(s):
    """
    Plotting-only helper.
    """
    s = str(s).lower()
    if "sarcomere" in s:
        return "Sarcomere"
    if "mitolysosome" in s:
        return "Mitolysosome"
    if "mitophagophore" in s:
        return "Mitophagophore"
    if "mitochond" in s or "mito" in s:
        return "Mitochondria"
    return "Other"


# -------------------------------------------------------------
# Load CSV
# -------------------------------------------------------------
df = pd.read_csv(INPUT_CSV, low_memory=False)

print("\nLoaded rows:", len(df))
print("Loaded columns:", len(df.columns))

# -------------------------------------------------------------
# Find centroid columns
# -------------------------------------------------------------
if "centroid_x_px_reduced" in df.columns and "centroid_y_px_reduced" in df.columns:
    x_col = "centroid_x_px_reduced"
    y_col = "centroid_y_px_reduced"
elif "centroid_x_px_fullres" in df.columns and "centroid_y_px_fullres" in df.columns:
    x_col = "centroid_x_px_fullres"
    y_col = "centroid_y_px_fullres"
else:
    raise ValueError("Could not find centroid_x_px_reduced/fullres and centroid_y_px_reduced/fullres columns.")

print("Using centroid columns:", x_col, y_col)

# -------------------------------------------------------------
# Preserve original assignments
# -------------------------------------------------------------
df["cell_label_id_original"] = pd.to_numeric(df["cell_label_id"], errors="coerce")

# -------------------------------------------------------------
# Infer field IDs
# -------------------------------------------------------------
df["field_id_raw"] = df.apply(infer_field_id_from_row, axis=1)

bad_field_mask = df["field_id_raw"].isna()
bad_count = int(bad_field_mask.sum())
if bad_count > 0:
    print(f"\nWarning: {bad_count} rows are missing a direct field_id and will be repaired by neighbor fill.")
    display(
        df.loc[bad_field_mask, [
            c for c in ["sample", "label_id", "source_label_file", "source_stack_file", "source_cell_file"]
            if c in df.columns
        ]].head(10)
    )

# Fill malformed rows from neighboring valid rows.
df["field_id"] = df["field_id_raw"].ffill().bfill()

if df["field_id"].isna().any():
    bad = df[df["field_id"].isna()]
    raise ValueError(
        "Some rows still do not have a field_id after forward/back fill.\n"
        f"Example rows:\n{bad.head(5)}"
    )

print("\nDetected fields:")
print(sorted(df["field_id"].dropna().unique()))

# -------------------------------------------------------------
# Correct assignments field-by-field
# -------------------------------------------------------------
field_cache = {}
corrected_ids = []
statuses = []
pixel_counts = []

for field_id, sub_idx in df.groupby("field_id").groups.items():
    cell_path = find_cell_tiff(field_id)
    print(f"Loading {field_id} -> {cell_path.name}")

    cell_fullres, cell_small = load_cell_labels(cell_path, target_shape=(TARGET_H, TARGET_W))
    field_cache[field_id] = (cell_fullres, cell_small, cell_path)

    for idx in sub_idx:
        x = df.at[idx, x_col]
        y = df.at[idx, y_col]

        cell_id, status, pixel_count = assign_cell_from_centroid(cell_small, x, y)

        corrected_ids.append((idx, cell_id))
        statuses.append((idx, status))
        pixel_counts.append((idx, pixel_count))


# -------------------------------------------------------------
# Apply corrected values
# -------------------------------------------------------------
df["cell_label_id_corrected"] = np.nan
df["cell_region_pixel_count_corrected"] = np.nan

# Important: make this an object column so it can store strings
df["cell_assignment_status"] = pd.Series([None] * len(df), index=df.index, dtype="object")
df["cell_assignment_method_corrected"] = pd.Series([None] * len(df), index=df.index, dtype="object")

for idx, cell_id in corrected_ids:
    df.at[idx, "cell_label_id_corrected"] = int(cell_id)

for idx, status in statuses:
    df.at[idx, "cell_assignment_status"] = str(status)

for idx, pixel_count in pixel_counts:
    df.at[idx, "cell_region_pixel_count_corrected"] = int(pixel_count)
    df.at[idx, "cell_assignment_method_corrected"] = "cell_small_centroid_sample"

df["cell_label_id"] = df["cell_label_id_corrected"].fillna(0).astype(int)
df["cell_region_pixel_count"] = df["cell_region_pixel_count_corrected"].fillna(0).astype(int)
df["cell_normalization_method"] = np.where(df["cell_label_id"] > 0, "cell_small_centroid_sample", "none")


# -------------------------------------------------------------
# Mark changed rows
# -------------------------------------------------------------
orig_ids = pd.to_numeric(df["cell_label_id_original"], errors="coerce").fillna(0).astype(int)
df["cell_assignment_changed"] = orig_ids != df["cell_label_id"].astype(int)

# -------------------------------------------------------------
# Summary
# -------------------------------------------------------------
print("\n=================================================")
print("Correction summary")
print("=================================================")
print("Total rows            :", len(df))
print("Changed assignments   :", int(df["cell_assignment_changed"].sum()))
print("Unchanged assignments :", int((~df["cell_assignment_changed"]).sum()))
print("\nAssignment status counts:")
print(df["cell_assignment_status"].value_counts(dropna=False))

review_cols = [
    c for c in [
        "field_id",
        "label_id",
        "source_label_file",
        "cell_label_id_original",
        "cell_label_id",
        "cell_assignment_status",
        x_col,
        y_col,
    ]
    if c in df.columns
]

print("\nExample changed rows:")
display(df.loc[df["cell_assignment_changed"], review_cols].head(25))


# -------------------------------------------------------------
# Repair sample names of appended rows to include area
# ---------------------------------------------------------
def repair_sample_name(row):

    sample = str(row["sample"])
    source = str(row["source_label_file"])

    # If sample already has _Ar, keep it unchanged
    if "_Ar" in sample:
        return sample

    # Extract pattern like S2_Ar5 from source_label_file
    match = re.search(r"(S\d+_Ar\d+)", source)

    if match:
        return match.group(1)

    # Otherwise leave unchanged
    return sample

# ---------------------------------------------------------
# Apply repair
# ---------------------------------------------------------
df["sample_original"] = df["sample"]

df["sample"] = df.apply(repair_sample_name, axis=1)


# ---------------------------------------------------------
# Show examples of repaired rows
# ---------------------------------------------------------
changed = df[df["sample"] != df["sample_original"]]

print(f"\nRows repaired: {len(changed)}")

display(
    changed[
        ["sample_original", "sample", "source_label_file"]
    ].head(20)
)


# -------------------------------------------------------------
# Update group names for appended rows
# -------------------------------------------------------------

key_path = Path("key.csv")

key_df = pd.read_csv(key_path, low_memory=False).copy()

df = pd.merge(df.drop(columns='experiment_name'), key_df[['sample','experiment_name']], on='sample', how='left')


# -------------------------------------------------------------
# Save corrected CSV
# -------------------------------------------------------------
df.to_csv(OUTPUT_CSV, index=False)
print("\nSaved corrected CSV:")
print(OUTPUT_CSV.resolve())